# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 파리입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 파리입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도 도시는 파리입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 사업화 리뷰가 긍정인지 부정인지 판정해줘.
- "오히려 지금은 특허 회피를 위한 사업 구조 설계와 실증 데이터 확보가 더 중요해 보인다."
- "최고의 사업, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 의견으로 보입니다. 특허 회피와 실증 데이터 확보의 중요성을 강조하고 있기 때문입니다. 

두 번째 리뷰는 긍정적인 의견입니다. "최고의 사업"이라는 표현이 긍정적인 평가를 나타내고 있습니다. 

따라서, 첫 번째 리뷰는 부정적, 두 번째 리뷰는 긍정적이라고 판별할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "오히려 지금은 특허 회피를 위한 사업 구조 설계와 실증 데이터 확보가 더 중요해 보인다." → 긍정 | 7/10
입력: "확장성은 있다" → 부정 | 4/10

[실제 데이터]
- "사업성은 있는데 특허가 약하다."
- "사업성도 약하고 기술성도 공격받는다."
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "사업성은 있는데 특허가 약하다." → 긍정 | 6/10  
입력: "사업성도 약하고 기술성도 공격받는다." → 부정 | 3/10


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

KeyboardInterrupt: 

In [ ]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

---
## 2. 역할(Role) 설정 

In [6]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 일반적으로 대부분의 사람들에게 괜찮습니다. 연구에 따르면, 성인은 하루에 400mg 이하의 카페인을 섭취하는 것이 안전하다고 여겨지며, 이는 대략 커피 4잔에 해당합니다. 하지만 개인의 카페인 민감도, 건강 상태, 그리고 다른 음료나 음식에서 섭취하는 카페인 양에 따라 다를 수 있습니다.

카페인은 각성 효과가 있어 피로를 줄이고 집중력을 높이는 데 도움을 줄 수 있지만, 과도한 섭취는 불안, 불면증, 심장 두근거림 등의 부작용을 초래할 수 있습니다. 따라서 자신의 몸 상태를 잘 살피고, 필요에 따라 조절하는 것이 중요합니다. 만약 특정 건강 문제가 있거나 카페인에 민감하다면, 전문가와 상담하는 것이 좋습니다.




In [7]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 보통 커피 한 잔에는 약 95mg의 카페인이 들어있습니다. 따라서 하루 4잔을 마신다면 약 380mg의 카페인을 섭취하게 됩니다. 

미국 식품의약국(FDA)에서는 건강한 성인이 하루에 최대 400mg의 카페인을 섭취하는 것이 안전하다고 권장하고 있습니다. 하지만 개인의 카페인 내성이나 건강 상태에 따라 다를 수 있으므로, 다음과 같은 점을 고려하는 것이 좋습니다:

1. **카페인 민감성**: 카페인에 민감한 사람은 불안감, 불면증, 심박수 증가 등의 증상을 경험할 수 있습니다.
2. **수분 섭취**: 커피는 이뇨 작용이 있을 수 있으므로 충분한 수분을 섭취하는 것이 중요합니다.
3. **건강 상태**: 심장 질환이나 불안 장애가 있는 경우, 카페인 섭취를 줄이는 것이 좋습니다.
4. **임신 중**: 임신 중인 경우 카페인 섭취를 제한하는 것이 권장됩니다.

결론적으로, 하루 4잔의 커피는 대부분의 사람에게 괜찮지만, 개인의 상황에 따라 조절하는 것이 중요합니다. 필요하다면 전문가와 상담하는 것도 좋은 방법입니다.


In [ ]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 안녕하세요! 인공지능(AI)은 컴퓨터 시스템이 인간의 인지 기능을 모방하여 학습, 추론, 문제 해결 등을 수행할 수 있도록 하는 기술입니다. AI는 다양한 형태로 존재하며, 주로 머신러닝, 딥러닝, 자연어 처리 등 여러 분야로 나뉩니다.

예를 들어, 머신러닝은 데이터를 분석하여 패턴을 인식하고, 이를 바탕으로 예측을 수행하는 방법입니다. 딥러닝은 이러한 머신러닝의 한 분야로, 인공 신경망을 사용하여 더욱 복잡한 데이터에서 학습할 수 있게 해줍니다. 자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술로, 챗봇이나 음성 인식 시스템에 활용됩니다.

인공지능은 의료, 금융, 물류, 고객 서비스 등 다양한 분야에서 활용되고 있으며, 우리의 일상생활에 점점 더 많은 영향을 미치고 있습니다. 더 궁금한 점이 있으시면 언제든지 말씀해 주세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해 및 자연어 처리와 같은 작업을 수행할 수 있도록 하는 기술입니다. 인공지능은 여러 분야에서 활용되며, 크게 두 가지 유형으로 나눌 수 있습니다.

1. **약한 인공지능(Weak AI)**: 특정 작업을 수행하도록 설계된 시스템으로, 제한된 범위 내에서만 기능합니다. 예를 들어, 음성 인식 소프트웨어나 추천 알고리즘이 이에 해당합니다.

2. **강한 인공지능(Strong AI)**: 인간과 유사한 지능을 갖춘 시스템으로, 스스로 학습하고 다양한 문제를 해결할 수 있는 능력을 지닙니다. 현재로서는 이론적인 개념에 가깝습니다.

인공지능의 주요 기술로는 머신러닝(Machine Learning), 딥러닝(Deep Learning), 자연어 처리(Natural Language Processing), 컴퓨터 비전(Computer Vision) 등이 있습니다. 머신러닝은 데이

: 

---
## 3. 출력 형식 지정 

In [3]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이함.
- **교통비 및 시간 절약**: 출퇴근 시간이 없으므로 교통비와 시간을 절약할 수 있음.
- **편안한 근무 환경**: 자신이 선호하는 환경에서 일할 수 있어 집중력과 생산성이 향상될 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있음.
- **업무와 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려질 수 있음.
- **자기 관리의 어려움**: 자율성이 높아지면서 스스로 동기를 부여하고 시간을 관리하는 것이 어려울 수 있음.




In [4]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목     | 장점                           | 단점                           |
|----------|--------------------------------|--------------------------------|
| 유연성   | 근무 시간을 자유롭게 조정 가능 | 일과 개인 생활의 경계 모호함  |
| 교통비 절감 | 출퇴근 시간과 비용 절약       | 사회적 상호작용 부족          |
| 생산성   | 집중할 수 있는 환경 조성 가능  | 자칫 산만해질 수 있음         |


---
### 출력 형식 (표 / JSON)

In [5]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인           | 설명                                   | 대응                       |
|----------------|----------------------------------------|----------------------------|
| 화학물질 순도  | 사용되는 화학물질의 순도가 낮으면 불순물이 생겨 수율이 감소할 수 있음 | 고순도 화학물질 사용       |
| 공정 온도      | 식각 공정의 온도가 적절하지 않으면 식각 속도나 균일성이 저하될 수 있음 | 온도 모니터링 및 조절      |
| 마스크 품질    | 마스크의 품질이 낮으면 식각 시 패턴이 손상되거나 왜곡될 수 있음 | 고품질 마스크 사용 및 관리 |


In [6]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "화학물질 농도",
      "action": "적절한 농도를 유지하여 식각 속도와 균일성을 확보"
    },
    {
      "name": "식각 시간",
      "action": "최적의 식각 시간을 설정하여 과식각이나 부족식각 방지"
    },
    {
      "name": "온도",
      "action": "온도를 일정하게 유지하여 반응 속도와 품질을 향상"
    }
  ]
}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [ ]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

In [ ]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

---
## 5. System + User 프롬프트 

In [7]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 월요일 기획안 제출을 우선으로 하고, 수요일 코드 리뷰를 금요일로 미루세요.

- 월요일: 기획안 제출 (우선순위 1)
- 수요일: 코드 리뷰 (금요일로 변경)
- 금요일: 발표 (변경 없음)


In [8]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론 | 리스크 | 가정 |
|------|--------|------|
| VIP 이탈 상위 200명에 대해 맞춤형 리워드 프로그램 및 직접적인 커뮤니케이션을 통해 재유치를 목표로 한다. | - 리워드 프로그램이 효과를 발휘하지 못할 경우 고객 이탈이 지속될 수 있음.<br>- 직접 커뮤니케이션이 고객에게 부담으로 작용할 가능성 있음. | - 고객들이 리워드 프로그램에 긍정적인 반응을 보일 것.<br>- 직접적인 소통을 통해 고객의 피드백을 수용할 수 있는 여지가 있을 것. |

### 액션 플랜
| 단계 | 액션 아이템 | 세부 내용 |
|------|--------------|-----------|
| 1 | 데이터 분석 | 이탈 고객의 구매 패턴 및 선호도를 분석하여 맞춤형 리워드 제안 준비. |
| 2 | 리워드 프로그램 설계 | VIP 고객을 위한 특별 할인, 포인트 적립, 한정판 상품 제공 등 다양한 리워드 옵션 기획. |
| 3 | 커뮤니케이션 전략 | 개인화된 이메일 및 문자 메시지를 통해 리워드 프로그램 안내 및 고객의 소중함 강조. |
| 4 | 피드백 수집 | 고객의 반응을 모니터링하고, 필요시 리워드 프로그램 수정 및 보완. |
| 5 | 성과 분석 | 프로그램 시행 후 재유치율 및 고객 만족도 분석하여 향후 전략 수립에 반영. |


---
## 6. Self-check Prompting

In [9]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)


--- Self-check 후 최종 답변(수정본) ---
### 점검 결과
- 논리적 모순/비약이 없음.
- 실행 가능성이 높고 액션 아이템이 명확함.
- 가정이 다소 과도함. 고객 반응에 대한 일반화를 피해야 함.
- 표 형식이 지켜짐.

### 수정본
| 결론 | 리스크 | 가정 |
|------|--------|------|
| VIP 이탈 상위 200명에 대해 맞춤형 리워드 프로그램 및 직접적인 커뮤니케이션을 통해 재유치를 목표로 한다. | - 리워드 프로그램이 효과를 발휘하지 못할 경우 고객 이탈이 지속될 수 있음.<br>- 직접 커뮤니케이션이 고객에게 부담으로 작용할 가능성 있음. | - 고객들이 리워드 프로그램에 긍정적인 반응을 보일 것.<br>- 고객의 피드백을 수용할 수 있는 여지가 있을 것. |

### 액션 플랜
| 단계 | 액션 아이템 | 세부 내용 |
|------|--------------|-----------|
| 1 | 데이터 분석 | 이탈 고객의 구매 패턴 및 선호도를 분석하여 맞춤형 리워드 제안 준비. |
| 2 | 리워드 프로그램 설계 | VIP 고객을 위한 특별 할인, 포인트 적립, 한정판 상품 제공 등 다양한 리워드 옵션 기획. |
| 3 | 커뮤니케이션 전략 | 개인화된 이메일 및 문자 메시지를 통해 리워드 프로그램 안내 및 고객의 소중함 강조. |
| 4 | 피드백 수집 | 고객의 반응을 모니터링하고, 필요시 리워드 프로그램 수정 및 보완. |
| 5 | 성과 분석 | 프로그램 시행 후 재유치율 및 고객 만족도 분석하여 향후 전략 수립에 반영. |


---
## 7. Constraint Prompting

In [10]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)


--- 조건 기반 최종 답변(답변만) ---
결론: VIP 이탈 방지를 위한 액션 플랜은 맞춤형 프로모션과 고객 관계 강화로 구성된다.

| 타깃         | 액션                             | 채널               | 우선순위 | 기간         | 기대효과                          | 담당        |
|--------------|----------------------------------|-------------------|----------|--------------|-----------------------------------|-------------|
| 상위 200명   | 맞춤형 할인 쿠폰 제공           | 이메일            | 높음     | 1주 이내    | 재구매 유도 및 고객 충성도 증가   | 마케팅팀    |
| 상위 200명   | 특별한 상품 추천 리스트 발송    | 이메일            | 중간     | 1주 이내    | 개인화된 경험 제공 및 관심 재유도 | MD팀        |
| 상위 200명   | VIP 전용 이벤트 초대            | 문자               | 높음     | 2주 이내    | 고객 소속감 증대 및 참여 유도     | 이벤트팀    |
| 상위 200명   | 고객 만족도 설문 조사           | 온라인 설문조사   | 중간     | 2주 이내    | 고객 피드백 수집 및 개선점 도출   | CS팀        |
| 상위 200명   | 개인화된 뉴스레터 발송          | 이메일            | 낮음     | 2주 이내    | 지속적인 관계 유지                 | 마케팅팀    |

### 리스크
- 고객 개인정보 유출 및 컴플라이언스 위반 리스크
- 맞춤형 프로모션이 오히려 반감을 살 가능성

### 가정
- VIP 고객의 반응이 긍정적일